In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()



Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dl_lab_3_product_segmentation_path = kagglehub.competition_download('dl-lab-3-product-segmentation')
print('Data source import complete.')



100%|██████████| 191M/191M [00:02<00:00, 93.0MB/s]

Extracting files...


Data source import complete.


In [3]:
pip install segmentation-models-pytorch timm opencv-python pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.0 MB/s eta 0:00:00


In [4]:
#pip uninstall -y torch torchvision torchaudio

In [ ]:
#pip install --no-cache-dir torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn
import albumentations as A
import gc


# =========================
# CONFIG (УЛУЧШЕННАЯ)
# =========================

DATA_ROOT = Path("/root/.cache/kagglehub/competitions/dl-lab-3-product-segmentation/train")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"

SAVE_DIR = Path("./seg_train_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 512
BATCH_SIZE = 8
NUM_EPOCHS = 30
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.2
NUM_WORKERS = 2
SEED = 42
THRESHOLD = 0.5

MODEL_NAME = "UnetPlusPlus"
ENCODER_NAME = "efficientnet-b2"
ENCODER_WEIGHTS = "imagenet"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================
# UTILS
# =========================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def clean_memory():
    """Очистка памяти GPU"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def find_image_for_stem(images_dir: Path, stem: str) -> Path | None:
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def extract_edges(mask_numpy: np.ndarray, thickness: int = 3) -> np.ndarray:
    mask_uint8 = (mask_numpy * 255).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (thickness, thickness))
    dilated = cv2.dilate(mask_uint8, kernel, iterations=1)
    eroded = cv2.erode(mask_uint8, kernel, iterations=1)
    morph_edges = (dilated - eroded) > 0

    canny_edges = cv2.Canny(mask_uint8, 50, 150) > 0

    edges = morph_edges | canny_edges
    return edges.astype(np.float32)


# =========================
# DATASET with EDGES (УЛУЧШЕННЫЙ)
# =========================
class BinarySegDataset(Dataset):
    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        img_size: int = 512,
        encoder_name: str = "efficientnet-b2",
        encoder_weights: str | None = "imagenet",
        augment: bool = False,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.img_size = img_size
        self.augment = augment

        self.preprocess_input = None
        if encoder_weights is not None:
            self.preprocess_input = get_preprocessing_fn(encoder_name, pretrained=encoder_weights)

        self.samples = []
        for mask_path in sorted(self.masks_dir.glob("*.png")):
            stem = mask_path.stem
            image_path = find_image_for_stem(self.images_dir, stem)
            if image_path is not None:
                self.samples.append((image_path, mask_path))

        if not self.samples:
            raise RuntimeError(f"No paired image/mask samples found")

        print(f"Found {len(self.samples)} paired samples (augment={augment})")

        self.train_transform = A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.3),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.CoarseDropout(
                max_holes=8,
                max_height=64,
                max_width=64,
                fill_value=0,
                p=0.5
            ),
        ])

        self.val_transform = A.Compose([
            A.Resize(img_size, img_size),
        ])

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, mask_path = self.samples[idx]

        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise RuntimeError(f"Cannot read image: {image_path}")
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Cannot read mask: {mask_path}")
        mask = (mask > 0).astype(np.float32)

        transform = self.train_transform if self.augment else self.val_transform
        augmented = transform(image=image_rgb, mask=mask)
        image_rgb = augmented['image']
        mask = augmented['mask']
        edges = extract_edges(mask, thickness=5)

        image_rgb = image_rgb.astype(np.float32)
        if self.preprocess_input is not None:
            image_rgb = self.preprocess_input(image_rgb)
        else:
            image_rgb = image_rgb / 255.0

        image = torch.from_numpy(image_rgb.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask).float().unsqueeze(0)
        edges = torch.from_numpy(edges).float().unsqueeze(0)

        target = torch.cat([mask, edges], dim=0)

        return image, target


# =========================
# LOSS FUNCTION (УЛУЧШЕННЫЙ)
# =========================
class MultiTaskLoss(nn.Module):
    def __init__(self, edge_weight: float = 1.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.focal = smp.losses.FocalLoss(mode='binary', alpha=0.75, gamma=2.0)
        self.edge_weight = edge_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        mask_logits = logits[:, 0:1].contiguous()
        edge_logits = logits[:, 1:2].contiguous()
        mask_target = targets[:, 0:1].contiguous()
        edge_target = targets[:, 1:2].contiguous()

        mask_loss = self.bce(mask_logits, mask_target) + self.dice(mask_logits, mask_target) + self.focal(mask_logits, mask_target)
        edge_loss = self.bce(edge_logits, edge_target)

        return mask_loss + self.edge_weight * edge_loss


# =========================
# MODEL
# =========================
def build_model() -> nn.Module:
    if MODEL_NAME == "Unet":
        model = smp.Unet(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=2,
            activation=None,
        )
    elif MODEL_NAME == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=2,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}")
    return model


# =========================
# TRAIN / VAL LOOPS (С TTA ДЛЯ ВАЛИДАЦИИ)
# =========================
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    running_loss = 0.0
    running_dice = 0.0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = loss_fn(logits, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        pred_mask = (torch.sigmoid(logits[:, 0:1]) > THRESHOLD).float()
        true_mask = targets[:, 0:1]
        intersection = (pred_mask * true_mask).sum()
        dice = (2.0 * intersection) / (pred_mask.sum() + true_mask.sum() + 1e-7)
        running_dice += dice.item()

    n = len(loader)
    return {"loss": running_loss / n, "dice": running_dice / n}


@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()
    running_loss = 0.0
    running_dice = 0.0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = loss_fn(logits, targets)

        running_loss += loss.item()

        pred_mask = (torch.sigmoid(logits[:, 0:1]) > THRESHOLD).float()
        true_mask = targets[:, 0:1]
        intersection = (pred_mask * true_mask).sum()
        dice = (2.0 * intersection) / (pred_mask.sum() + true_mask.sum() + 1e-7)
        running_dice += dice.item()

    n = len(loader)
    return {"loss": running_loss / n, "dice": running_dice / n}


# =========================
# MAIN (ИСПРАВЛЕННЫЙ)
# =========================
def main():

    clean_memory()
    seed_everything(SEED)

    full_dataset = BinarySegDataset(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR,
        img_size=IMG_SIZE,
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS,
        augment=False,
    )

    total_len = len(full_dataset)
    val_size = int(total_len * VAL_RATIO)
    train_size = total_len - val_size

    generator = torch.Generator().manual_seed(SEED)
    train_indices, val_indices = random_split(
        range(total_len),
        [train_size, val_size],
        generator=generator
    )

    train_dataset = BinarySegDataset(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR,
        img_size=IMG_SIZE,
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS,
        augment=True,
    )
    train_dataset.samples = [full_dataset.samples[i] for i in train_indices.indices]

    val_dataset = BinarySegDataset(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR,
        img_size=IMG_SIZE,
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS,
        augment=False,
    )
    val_dataset.samples = [full_dataset.samples[i] for i in val_indices.indices]

    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")


    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        drop_last=False,
    )

    model = build_model().to(DEVICE)
    loss_fn = MultiTaskLoss(edge_weight=1.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

    best_val_dice = -1.0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
        val_metrics = validate_one_epoch(model, val_loader, loss_fn, DEVICE)

        scheduler.step()

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_metrics["loss"],
            "train_dice": train_metrics["dice"],
            "val_loss": val_metrics["loss"],
            "val_dice": val_metrics["dice"],
        }
        history.append(row)

        print(
            f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
            f"train_loss={row['train_loss']:.4f} train_dice={row['train_dice']:.4f} | "
            f"val_loss={row['val_loss']:.4f} val_dice={row['val_dice']:.4f}"
        )

        if row["val_dice"] > best_val_dice:
            best_val_dice = row["val_dice"]
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_dice": row["val_dice"],
                    "config": {
                        "MODEL_NAME": MODEL_NAME,
                        "ENCODER_NAME": ENCODER_NAME,
                        "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                        "IMG_SIZE": IMG_SIZE,
                    },
                },
                SAVE_DIR / "best.pth",
            )
            print(f"✓ Saved new best model with val_dice={best_val_dice:.4f}")

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": row["val_dice"],
                "config": {
                    "MODEL_NAME": MODEL_NAME,
                    "ENCODER_NAME": ENCODER_NAME,
                    "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                    "IMG_SIZE": IMG_SIZE,
                },
            },
            SAVE_DIR / "last.pth",
        )

        clean_memory()

    import pandas as pd
    history_df = pd.DataFrame(history)
    history_df.to_csv(SAVE_DIR / "history.csv", index=False)
    print(f"\n Training finished! Best val_dice={best_val_dice:.4f}")


if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

Found 2000 paired samples (augment=False)
Found 2000 paired samples (augment=True)
Found 2000 paired samples (augment=False)
Train samples: 1600, Val samples: 400


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_31892/3721394286.py:122: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
/tmp/ipykernel_31892/3721394286.py:123: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(


model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

In [11]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn


# =========================
# CONFIG
# =========================
TEST_IMAGES_DIR = Path("/root/.cache/kagglehub/competitions/dl-lab-3-product-segmentation/test_images")
OUTPUT_CSV = "submission.csv"


# путь к вашему чекпоинту после обучения
CHECKPOINT_PATH = Path(r"./seg_train_runs/best.pth")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



# =========================
# HELPERS
# =========================
def cv2_imread_unicode(path: Path, flags=cv2.IMREAD_COLOR):
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, flags)


def build_model(model_name: str, encoder_name: str, encoder_weights=None):
    if model_name == "Unet":
        model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=2,
            activation=None,
        )
    elif model_name == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=2,
            activation=None,
        )
    elif model_name == "FPN":
        model = smp.FPN(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=2,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {model_name}")
    return model




def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.astype(np.uint8).tolist(), separators=(",", ":"))


# =========================
# LOAD CHECKPOINT
# =========================
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

if "config" not in checkpoint:
    raise ValueError("Checkpoint does not contain 'config'.")

cfg = checkpoint["config"]

MODEL_NAME = cfg["MODEL_NAME"]
ENCODER_NAME = cfg["ENCODER_NAME"]
ENCODER_WEIGHTS = cfg.get("ENCODER_WEIGHTS", None)
IMG_SIZE = int(cfg["IMG_SIZE"])

print("Loaded checkpoint config:")
print("MODEL_NAME     =", MODEL_NAME)
print("ENCODER_NAME   =", ENCODER_NAME)
print("ENCODER_WEIGHTS=", ENCODER_WEIGHTS)
print("IMG_SIZE       =", IMG_SIZE)

model = build_model(
    model_name=MODEL_NAME,
    encoder_name=ENCODER_NAME,
    encoder_weights=None,  # веса энкодера уже придут из checkpoint
)
model.load_state_dict(checkpoint["model_state_dict"], strict=False)
model.to(DEVICE)
model.eval()

preprocess_input = None
if ENCODER_WEIGHTS is not None:
    preprocess_input = get_preprocessing_fn(ENCODER_NAME, pretrained=ENCODER_WEIGHTS)


# =========================
# INFERENCE
# =========================
image_paths = sorted([p for p in TEST_IMAGES_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS])

if not image_paths:
    raise FileNotFoundError(f"No images found in: {TEST_IMAGES_DIR}")

print(f"Found {len(image_paths)} test images")

rows = []

with torch.no_grad():
    for i, img_path in enumerate(image_paths, 1):
        img_bgr = cv2_imread_unicode(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            print(f"[skip] cannot read: {img_path}")
            continue

        H, W = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # resize под вход модели
        inp = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        inp = inp.astype(np.float32)

        if preprocess_input is not None:
            inp = preprocess_input(inp)
        else:
            inp = inp / 255.0

        inp = torch.from_numpy(inp.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

        logits = model(inp)                      # [1, 2, H, W]
        mask_logits = logits[:, 0:1]             # [1, 1, H, W] - только маска
        probs = torch.sigmoid(mask_logits)[0, 0] # [H, W]
        pred = probs.detach().cpu().numpy()

        # возвращаем в исходный размер
        if pred.shape != (H, W):
            pred = cv2.resize(pred.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)

        mask = (pred > THRESHOLD).astype(np.uint8)

        rows.append(
            {
                "ImageId": img_path.name,
                "mask": serialize_mask(mask),
            }
        )

        if i % 100 == 0 or i == len(image_paths):
            print(f"Processed {i}/{len(image_paths)}")

submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print("Done.")
print(f"Saved submission to: {OUTPUT_CSV}")
print(submission_df.head())

Loaded checkpoint config:
MODEL_NAME     = UnetPlusPlus
ENCODER_NAME   = timm-efficientnet-b3
ENCODER_WEIGHTS= imagenet
IMG_SIZE       = 512
Found 2000 test images
Processed 100/2000
Processed 200/2000
Processed 300/2000
Processed 400/2000
Processed 500/2000
Processed 600/2000
Processed 700/2000
Processed 800/2000
Processed 900/2000
Processed 1000/2000
Processed 1100/2000
Processed 1200/2000
Processed 1300/2000
Processed 1400/2000
Processed 1500/2000
Processed 1600/2000
Processed 1700/2000
Processed 1800/2000
Processed 1900/2000
Processed 2000/2000
Done.
Saved submission to: submission.csv
                                             ImageId  \
0  10.107.215.111_20260119142748_a0be018c-d498-4b...   
1  10.107.215.111_20260119203335_58a6ef5c-c360-4e...   
2  10.107.215.111_20260119203341_bb74ac11-3f14-4a...   
3  10.107.215.111_20260122115937_db9b4af5-0f1e-42...   
4  10.107.215.111_20260124135952_6fd575ec-50e8-4f...   

                                                mask  
0  [[0,0,0,